In [1]:
# Load Model and Tokenizer

from transformers import GPT2Model, GPT2Tokenizer

# Load the pre-trained GPT-2 model and tokenizer
model_name = "gpt2"
model = GPT2Model.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

/Users/harshit/Documents/UWM/F25/839- FM/HW1/vhw1/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [2]:
# Parameter Counting Function
def count_params(module, is_human: bool = False):
    """Counts the number of trainable parameters in a PyTorch module."""
    params: int = sum(p.numel() for p in module.parameters() if p.requires_grad)
    if is_human:
        return f"{params / 1e6:.2f}M"
    return params

# Print the model structure to understand its components
print(model)

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


In [3]:
# Extract key dimensions from the model's configuration
V: int = model.config.vocab_size
E: int = model.config.hidden_size
L: int = model.config.n_layer
n_h: int = model.config.n_head
M: int = model.config.n_inner if model.config.n_inner is not None else 4 * E
max_pos: int = model.config.max_position_embeddings


In [4]:
# Analyze Word Token Embeddings (wte)
expected_wte = V * E
print(f"wte (word token embeddings) | Expected: {expected_wte}")
print(f"wte (word token embeddings) | True:     {count_params(model.wte)}")

wte (word token embeddings) | Expected: 38597376
wte (word token embeddings) | True:     38597376


In [5]:
# Analyze Word Position Embeddings (wpe)
expected_wpe = max_pos * E
print(f"wpe (position embeddings) | Expected: {expected_wpe}")
print(f"wpe (position embeddings) | True:     {count_params(model.wpe)}")

wpe (position embeddings) | Expected: 786432
wpe (position embeddings) | True:     786432


In [6]:
# Analyze Attention Block (Per Layer)

# Formula: (Hidden Size * 3 * Hidden Size) + (3 * Hidden Size)
expected_c_attn = (E * 3 * E) + (3 * E)
print(f"c_attn (QKV projection) | Expected: {expected_c_attn}")
print(f"c_attn (QKV projection) | True:     {count_params(model.h[0].attn.c_attn)}")

# Output Projection (c_proj)
# Formula: (Hidden Size * Hidden Size) + Hidden Size
expected_c_proj_attn = (E * E) + E
print(f"c_proj (Attention output) | Expected: {expected_c_proj_attn}")
print(f"c_proj (Attention output) | True:     {count_params(model.h[0].attn.c_proj)}")



c_attn (QKV projection) | Expected: 1771776
c_attn (QKV projection) | True:     1771776
c_proj (Attention output) | Expected: 590592
c_proj (Attention output) | True:     590592


In [7]:
# Analyze MLP Block (Per Layer)

# Formula: (Hidden Size * Intermediate Size) + Intermediate Size
expected_c_fc = (E * M) + M
print(f"c_fc (MLP up-projection) | Expected: {expected_c_fc}")
print(f"c_fc (MLP up-projection) | True:     {count_params(model.h[0].mlp.c_fc)}")

# Output Projection Layer (c_proj)
# Formula: (Intermediate Size * Hidden Size) + Hidden Size
expected_c_proj_mlp = (M * E) + E
print(f"c_proj (MLP down-projection) | Expected: {expected_c_proj_mlp}")
print(f"c_proj (MLP down-projection) | True:     {count_params(model.h[0].mlp.c_proj)}")




c_fc (MLP up-projection) | Expected: 2362368
c_fc (MLP up-projection) | True:     2362368
c_proj (MLP down-projection) | Expected: 2360064
c_proj (MLP down-projection) | True:     2360064


In [8]:
# Analyze Layer Normalization

# LayerNorm parameters (weight + bias) = 2 * Hidden Size
expected_ln = 2 * E

# Input LayerNorm (ln_1)
print(f"ln_1 (Input LayerNorm) | Expected: {expected_ln}")
print(f"ln_1 (Input LayerNorm) | True:     {count_params(model.h[0].ln_1)}")

# Post-Attention LayerNorm (ln_2)
print(f"ln_2 (Post-Attention LayerNorm) | Expected: {expected_ln}")
print(f"ln_2 (Post-Attention LayerNorm) | True:     {count_params(model.h[0].ln_2)}")

# Final LayerNorm (ln_f)
print(f"ln_f (Final LayerNorm) | Expected: {expected_ln}")
print(f"ln_f (Final LayerNorm) | True:     {count_params(model.ln_f)}")

ln_1 (Input LayerNorm) | Expected: 1536
ln_1 (Input LayerNorm) | True:     1536
ln_2 (Post-Attention LayerNorm) | Expected: 1536
ln_2 (Post-Attention LayerNorm) | True:     1536
ln_f (Final LayerNorm) | Expected: 1536
ln_f (Final LayerNorm) | True:     1536


In [9]:
# Calculate Total Parameters

# Sum of all components: Embeddings + (Layers * Params_per_layer) + Final_LayerNorm
params_per_layer = (
    expected_c_attn +
    expected_c_proj_attn +
    expected_c_fc +
    expected_c_proj_mlp +
    2*expected_ln
)

expected_total = (
    expected_wte +
    expected_wpe +
    (L * params_per_layer) +
    expected_ln
)

print(f"Number of Layers (L): {L}")
print(f"Total | Expected: {expected_total}")
print(f"Total | True:     {count_params(model)}")
print(f"Total (Human Readable) | True: {count_params(model, is_human=True)}")

Number of Layers (L): 12
Total | Expected: 124439808
Total | True:     124439808
Total (Human Readable) | True: 124.44M
